# Séance 5 · Exercices — Analyser et raconter une histoire avec les données · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

**Niveau de la séance : ⭐⭐ Intermédiaire** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

- Comment travailler : lis l'énoncé, code dans la cellule « À toi », lance la cellule de vérification (✅ / ❌), et n'ouvre la solution qu'après avoir vraiment essayé.
- Ce notebook tourne dans **Google Colab** : rien à installer.
- Clique sur une cellule et fais `Maj + Entrée` pour l'exécuter. Fais les exercices dans l'ordre : certains réutilisent les variables des précédents.


## Préparation

Trois jeux de données : **Tips** (244 additions d'un restaurant) et les **Pokémon**, comme dans la leçon, plus un nouveau : le catalogue **Netflix** (7 787 films et séries : titre, type, pays, année de sortie, durée, genres...). Les graphiques Plotly (`px`) sont interactifs : survole, zoome, clique sur la légende.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

URL_TIPS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
URL_POKEMON = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
URL_NETFLIX = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-04-20/netflix_titles.csv"
try:
    tips = pd.read_csv(URL_TIPS)
    pokemon = pd.read_csv(URL_POKEMON)
    netflix = pd.read_csv(URL_NETFLIX)
    print("Tips :", tips.shape, "· Pokémon :", pokemon.shape, "· Netflix :", netflix.shape)
except Exception as erreur:
    print("Pas de réseau ? Impossible de charger les fichiers :", erreur)

stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
netflix.head(3)

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = booléen, ou fonction sans argument)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as erreur:
        print(f"❌ {nom} : erreur pendant la vérification → {erreur}")
        return False
    print(f"✅ {nom}" if ok else f"❌ {nom} : pas encore, réessaie !")
    return ok


## Exercice 1 ⭐ · Trois bonnes questions

Tu es data scientist pour le patron du restaurant. Écris **3 questions précises** dans la liste `questions`
(chacune finit par un `?` et peut se vérifier avec les colonnes de `tips`). Puis réponds à la première avec un
`groupby` : `pourboire_par_jour` (moyenne de `tip` par `day`, arrondie à 2 décimales) et `jour_le_plus_genereux`.

Résultat attendu : par exemple `Sat 2.99`, et un nom de jour.

<details><summary>Indice</summary>

`tips.groupby("day")["tip"].mean().round(2)` ; le jour du maximum : `.idxmax()`.

</details>

In [ ]:
# À toi
questions = ["", "", ""]
pourboire_par_jour = None
jour_le_plus_genereux = None
print(pourboire_par_jour)
print("Jour le plus généreux :", jour_le_plus_genereux)

In [ ]:
verifier("Exercice 1 · trois questions", len(questions) == 3 and all(isinstance(q, str) and q.strip().endswith("?") and len(q.strip()) > 15 for q in questions))
verifier("Exercice 1 · pourboire par jour", pourboire_par_jour is not None and float(pourboire_par_jour["Sun"]) == 3.26)
verifier("Exercice 1 · jour le plus généreux", jour_le_plus_genereux == "Sun")

<details><summary>Solution</summary>

```python
questions = ["Le pourboire moyen change-t-il selon le jour de la semaine ?",
             "Les fumeurs laissent-ils un pourboire plus gros que les non-fumeurs ?",
             "Le pourboire par personne baisse-t-il quand la table est grande ?"]
pourboire_par_jour = tips.groupby("day")["tip"].mean().round(2)
jour_le_plus_genereux = pourboire_par_jour.idxmax()
print(pourboire_par_jour)
print("Jour le plus généreux :", jour_le_plus_genereux)
```

</details>

## Exercice 2 ⭐ · Fumeurs ou non ?

Deuxième question : les fumeurs laissent-ils plus ? Calcule `pourboire_par_fumeur` (moyenne de `tip` selon `smoker`,
arrondie à 2 décimales), puis `difference` (l'écart absolu entre les deux moyennes, arrondi à 2 décimales).
Enfin, conclus dans `conclusion` : `"négligeable"` ou `"importante"` ?

Résultat attendu : une différence de quelques centimes.

<details><summary>Indice</summary>

`groupby("smoker")["tip"].mean()` ; les deux valeurs sont `serie["Yes"]` et `serie["No"]` ; `abs(a - b)`.

</details>

In [ ]:
# À toi
pourboire_par_fumeur = None
difference = None
conclusion = "..."   # "négligeable" ou "importante"
print(pourboire_par_fumeur)
print("Différence :", difference, "→", conclusion)

In [ ]:
verifier("Exercice 2 · moyennes", pourboire_par_fumeur is not None and float(pourboire_par_fumeur["Yes"]) == 3.01)
verifier("Exercice 2 · différence", difference == 0.02)
verifier("Exercice 2 · conclusion", conclusion == "négligeable")

<details><summary>Solution</summary>

```python
pourboire_par_fumeur = tips.groupby("smoker")["tip"].mean().round(2)
difference = round(abs(pourboire_par_fumeur["Yes"] - pourboire_par_fumeur["No"]), 2)
conclusion = "négligeable"   # 2 centimes d'écart sur 3 $, ça ne veut rien dire
print(pourboire_par_fumeur)
print("Différence :", difference, "→", conclusion)
```

</details>

## Exercice 3 ⭐ · Une tendance par génération

Les Pokémon sont-ils de plus en plus forts au fil des générations ? Calcule `total_par_generation` (moyenne de `Total`
par `Generation`, arrondie à 1 décimale), mets dans `generation_la_plus_forte` le numéro de la génération la plus forte,
et trace un graphique en barres avec un titre qui **donne la réponse** (pas « Total par génération »).

Résultat attendu : `total_par_generation[1]` vaut 426.8.

<details><summary>Indice</summary>

`pokemon.groupby("Generation")["Total"].mean().round(1)` puis `.plot(kind="bar", title=...)`.

</details>

In [ ]:
# À toi
total_par_generation = None
generation_la_plus_forte = None
print(total_par_generation)
print("Génération la plus forte :", generation_la_plus_forte)

In [ ]:
verifier("Exercice 3 · moyennes", total_par_generation is not None and float(total_par_generation[1]) == 426.8)
verifier("Exercice 3 · génération la plus forte", generation_la_plus_forte is not None and int(generation_la_plus_forte) == 4)

**Check-list** (pas de vérification automatique possible pour un graphique) :

- [ ] Le graphique en barres a un titre qui donne la réponse (« La génération 4 est la plus puissante, mais l'écart est faible »)
- [ ] L'axe vertical part de zéro (sinon l'écart entre générations paraît énorme)

<details><summary>Solution</summary>

```python
total_par_generation = pokemon.groupby("Generation")["Total"].mean().round(1)
generation_la_plus_forte = int(total_par_generation.idxmax())
print(total_par_generation)
print("Génération la plus forte :", generation_la_plus_forte)
total_par_generation.plot(kind="bar", figsize=(7, 4), title="La génération 4 est la plus puissante, mais l'écart reste faible")
plt.ylabel("Total moyen"); plt.ylim(0, 500); plt.show()
```

</details>

## Exercice 4 ⭐ · Bienvenue chez Netflix

Fais connaissance avec le catalogue `netflix` : `nb_films` et `nb_series` (colonne `type` : `Movie` ou `TV Show`),
`part_films` (le pourcentage de films, arrondi à 1 décimale) et `classification_top` (la valeur la plus fréquente
de `rating`, la classification du public visé : tout public, adultes...).

Résultat attendu : plus des deux tiers du catalogue sont des films.

<details><summary>Indice</summary>

`netflix["type"].value_counts()` ; `round(nb_films / len(netflix) * 100, 1)` ; `.value_counts().idxmax()`.

</details>

In [ ]:
# À toi
nb_films = None
nb_series = None
part_films = None
classification_top = None
print(nb_films, "films ·", nb_series, "séries ·", part_films, "% de films · classification la plus fréquente :", classification_top)

In [ ]:
verifier("Exercice 4 · films et séries", nb_films == 5377 and nb_series == 2410)
verifier("Exercice 4 · part des films", part_films == 69.1)
verifier("Exercice 4 · classification", classification_top == "TV-MA")

<details><summary>Solution</summary>

```python
comptes = netflix["type"].value_counts()
nb_films = int(comptes["Movie"])
nb_series = int(comptes["TV Show"])
part_films = round(nb_films / len(netflix) * 100, 1)
classification_top = netflix["rating"].value_counts().idxmax()
print(nb_films, "films ·", nb_series, "séries ·", part_films, "% de films · classification la plus fréquente :", classification_top)
```

</details>

## Exercice 5 ⭐⭐ · Les pourboires hors norme

Applique la règle des 1,5 écarts sur la colonne `tip` : `q1`, `q3`, `limite_haute = q3 + 1.5 × (q3 − q1)`,
puis `anomalies` (les lignes au-dessus de la limite), `nb_anomalies`, `plus_gros_pourboire` et `addition_du_plus_gros`
(le `total_bill` de cette addition). Trace aussi la boîte à moustaches de `tip`.

Résultat attendu : une petite dizaine d'anomalies, et un pourboire record à deux chiffres.

<details><summary>Indice</summary>

`tips["tip"].quantile([0.25, 0.75])` donne les deux quarts ; `tips[tips["tip"] > limite_haute]` ; `.sort_values("tip", ascending=False).iloc[0]`.

</details>

In [ ]:
# À toi
q1, q3 = None, None
limite_haute = None
anomalies = None
nb_anomalies = None
plus_gros_pourboire = None
addition_du_plus_gros = None
print("Limite :", limite_haute, "·", nb_anomalies, "anomalies · record :", plus_gros_pourboire, "$ sur une addition de", addition_du_plus_gros, "$")

In [ ]:
verifier("Exercice 5 · limite", limite_haute is not None and round(limite_haute, 3) == 5.906)
verifier("Exercice 5 · nombre d'anomalies", nb_anomalies == 9 and anomalies is not None and len(anomalies) == 9)
verifier("Exercice 5 · le record", plus_gros_pourboire == 10.0 and addition_du_plus_gros is not None and round(addition_du_plus_gros, 2) == 50.81)

<details><summary>Solution</summary>

```python
q1, q3 = tips["tip"].quantile([0.25, 0.75])
limite_haute = q3 + 1.5 * (q3 - q1)
anomalies = tips[tips["tip"] > limite_haute]
nb_anomalies = len(anomalies)
record = anomalies.sort_values("tip", ascending=False).iloc[0]
plus_gros_pourboire = record["tip"]
addition_du_plus_gros = record["total_bill"]
print("Limite :", limite_haute, "·", nb_anomalies, "anomalies · record :", plus_gros_pourboire, "$ sur une addition de", addition_du_plus_gros, "$")
plt.figure(figsize=(6, 3)); sns.boxplot(data=tips, x="tip"); plt.title("Les points isolés à droite sont les anomalies"); plt.show()
```

</details>

## Exercice 6 ⭐⭐ · Qui bouge avec qui ?

Calcule `corr_addition_pourboire` (corrélation entre `total_bill` et `tip`, arrondie à 2 décimales). Puis, sur les
6 stats des Pokémon (`stats`), calcule la matrice `matrice = pokemon[stats].corr()` et trouve `paire_la_plus_correlee`
et `paire_la_moins_correlee` (deux tuples de 2 noms de stats). Trace la heatmap.

Résultat attendu : l'addition et le pourboire sont nettement corrélés ; la paire la moins corrélée est presque à zéro.

<details><summary>Indice</summary>

Pour lister toutes les paires sans la diagonale : `paires = matrice.where(np.triu(np.ones(matrice.shape), k=1).astype(bool)).stack().sort_values()` ; la première est la moins corrélée, la dernière la plus.

</details>

In [ ]:
# À toi
corr_addition_pourboire = None
matrice = None
paire_la_plus_correlee = None
paire_la_moins_correlee = None
print("Addition / pourboire :", corr_addition_pourboire)
print("Plus corrélées :", paire_la_plus_correlee, "· moins corrélées :", paire_la_moins_correlee)

In [ ]:
verifier("Exercice 6 · addition / pourboire", corr_addition_pourboire == 0.68)
verifier("Exercice 6 · paire la plus corrélée", paire_la_plus_correlee is not None and set(paire_la_plus_correlee) == {"Defense", "Sp. Def"})
verifier("Exercice 6 · paire la moins corrélée", paire_la_moins_correlee is not None and set(paire_la_moins_correlee) == {"Defense", "Speed"})

<details><summary>Solution</summary>

```python
corr_addition_pourboire = round(tips["total_bill"].corr(tips["tip"]), 2)
matrice = pokemon[stats].corr()
paires = matrice.where(np.triu(np.ones(matrice.shape), k=1).astype(bool)).stack().sort_values()
paire_la_moins_correlee = paires.index[0]
paire_la_plus_correlee = paires.index[-1]
print("Addition / pourboire :", corr_addition_pourboire)
print("Plus corrélées :", paire_la_plus_correlee, "· moins corrélées :", paire_la_moins_correlee)
plt.figure(figsize=(6.5, 5)); sns.heatmap(matrice, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1); plt.show()
```

</details>

## Exercice 7 ⭐⭐ · Combien durent les films Netflix ?

La colonne `duration` mélange `"93 min"` (films) et `"4 Seasons"` (séries). Garde seulement les films dans `films`
(une copie), crée la colonne `films["duree_min"]` en nombre entier, puis calcule `duree_moyenne` (arrondie à 1 décimale)
et `film_le_plus_long` (son titre). Trace un histogramme des durées.

Résultat attendu : une durée moyenne autour de 100 minutes, et un film record de plus de 5 heures.

<details><summary>Indice</summary>

`netflix[netflix["type"] == "Movie"].copy()` ; `.str.replace(" min", "").astype(int)` ; `films.loc[films["duree_min"].idxmax(), "title"]`.

</details>

In [ ]:
# À toi
films = None
duree_moyenne = None
film_le_plus_long = None
print("Durée moyenne :", duree_moyenne, "min · le plus long :", film_le_plus_long)

In [ ]:
verifier("Exercice 7 · colonne duree_min", isinstance(films, pd.DataFrame) and "duree_min" in films.columns and pd.api.types.is_integer_dtype(films["duree_min"]) and len(films) == 5377)
verifier("Exercice 7 · durée moyenne", duree_moyenne == 99.3)
verifier("Exercice 7 · le plus long", film_le_plus_long == "Black Mirror: Bandersnatch")

<details><summary>Solution</summary>

```python
films = netflix[netflix["type"] == "Movie"].copy()
films["duree_min"] = films["duration"].str.replace(" min", "").astype(int)
duree_moyenne = round(films["duree_min"].mean(), 1)
film_le_plus_long = films.loc[films["duree_min"].idxmax(), "title"]
print("Durée moyenne :", duree_moyenne, "min · le plus long :", film_le_plus_long)
films["duree_min"].plot(kind="hist", bins=40, figsize=(7, 3.5), title="La plupart des films durent entre 80 et 120 minutes"); plt.xlabel("Durée (min)"); plt.show()
```

</details>

## Exercice 8 ⭐⭐ · Netflix année après année

Compte les titres par année de sortie (`release_year`) dans `titres_par_annee` (trié par année), trouve `annee_record`
et `nb_annee_record`. Puis, avec `films` de l'exercice précédent, calcule `duree_par_decennie` : la durée moyenne des
films par décennie (1940, 1950, ... 2020), arrondie à l'unité. Les films raccourcissent-ils ? Réponds dans `tendance`
(`"plus courts"` ou `"plus longs"`). Trace la courbe des titres par année depuis 2000.

Résultat attendu : `duree_par_decennie[2010]` vaut 97.

<details><summary>Indice</summary>

`value_counts().sort_index()` ; la décennie : `films["release_year"] // 10 * 10`, à donner directement à `groupby`.

</details>

In [ ]:
# À toi
titres_par_annee = None
annee_record = None
nb_annee_record = None
duree_par_decennie = None
tendance = "..."   # "plus courts" ou "plus longs"
print("Année record :", annee_record, "avec", nb_annee_record, "titres")
print(duree_par_decennie)

In [ ]:
verifier("Exercice 8 · année record", annee_record == 2018 and nb_annee_record == 1121)
verifier("Exercice 8 · durée par décennie", duree_par_decennie is not None and int(duree_par_decennie[2010]) == 97 and int(duree_par_decennie[1960]) == 142)
verifier("Exercice 8 · tendance", tendance == "plus courts")

<details><summary>Solution</summary>

```python
titres_par_annee = netflix["release_year"].value_counts().sort_index()
annee_record = int(titres_par_annee.idxmax())
nb_annee_record = int(titres_par_annee.max())
duree_par_decennie = films.groupby(films["release_year"] // 10 * 10)["duree_min"].mean().round(0)
tendance = "plus courts"   # 142 min dans les années 60, 97 dans les années 2010
print("Année record :", annee_record, "avec", nb_annee_record, "titres")
print(duree_par_decennie)
titres_par_annee[titres_par_annee.index >= 2000].plot(figsize=(7, 3.5), marker="o", title="Le catalogue explose à partir de 2015, pic en 2018"); plt.ylabel("Titres sortis"); plt.show()
```

</details>

## Exercice 9 ⭐⭐ · Corrélation n'est pas causalité

Le tableau `ete` (120 journées simulées) montre que les ventes de glaces et les noyades bougent ensemble.
Calcule `corr_brute` (corrélation glaces / noyades, arrondie à 2 décimales) et nomme la `variable_cachee`.
Puis prouve-le : la colonne `tranche` découpe les journées en « froid / moyen / chaud ». Calcule `corr_par_tranche`
(un dictionnaire tranche → corrélation glaces / noyades **à l'intérieur** de la tranche, arrondie à 2 décimales).
Si la température explique tout, la corrélation s'effondre dès qu'on la fixe.

Résultat attendu : `corr_brute` élevée, et toutes les corrélations par tranche nettement plus faibles.

<details><summary>Indice</summary>

`ete["glaces"].corr(ete["noyades"])` ; pour les tranches : `for tranche, groupe in ete.groupby("tranche", observed=True):` puis la même corrélation sur `groupe`.

</details>

In [ ]:
# Les données de la leçon (fictives !) : la température cause tout, glaces et noyades ne se parlent jamais
rng = np.random.default_rng(0)
temperature = rng.uniform(5, 35, 120)
glaces = 20 + 8 * temperature + rng.normal(0, 25, 120)
noyades = 0.5 + 0.12 * temperature + rng.normal(0, 0.8, 120)
ete = pd.DataFrame({"temperature": temperature.round(1), "glaces": glaces.round(0), "noyades": noyades.clip(0).round(1)})
ete["tranche"] = pd.cut(ete["temperature"], bins=[0, 15, 25, 40], labels=["froid", "moyen", "chaud"])

# À toi
corr_brute = None
variable_cachee = "..."
corr_par_tranche = {}
print("Corrélation brute :", corr_brute, "· variable cachée :", variable_cachee)
print("Par tranche :", corr_par_tranche)

In [ ]:
verifier("Exercice 9 · corrélation brute", corr_brute == 0.72)
verifier("Exercice 9 · variable cachée", isinstance(variable_cachee, str) and variable_cachee.lower().startswith("temp"))
verifier("Exercice 9 · la corrélation s'effondre par tranche", len(corr_par_tranche) == 3 and all(v < 0.5 for v in corr_par_tranche.values()))

<details><summary>Solution</summary>

```python
rng = np.random.default_rng(0)
temperature = rng.uniform(5, 35, 120)
glaces = 20 + 8 * temperature + rng.normal(0, 25, 120)
noyades = 0.5 + 0.12 * temperature + rng.normal(0, 0.8, 120)
ete = pd.DataFrame({"temperature": temperature.round(1), "glaces": glaces.round(0), "noyades": noyades.clip(0).round(1)})
ete["tranche"] = pd.cut(ete["temperature"], bins=[0, 15, 25, 40], labels=["froid", "moyen", "chaud"])

corr_brute = round(ete["glaces"].corr(ete["noyades"]), 2)
variable_cachee = "temperature"
corr_par_tranche = {}
for tranche, groupe in ete.groupby("tranche", observed=True):
    corr_par_tranche[tranche] = round(groupe["glaces"].corr(groupe["noyades"]), 2)
print("Corrélation brute :", corr_brute, "· variable cachée :", variable_cachee)
print("Par tranche :", corr_par_tranche)   # à température (presque) fixe, plus grand-chose : la chaleur faisait tout
```

</details>

## Exercice 10 ⭐⭐ · Le graphique menteur

Le graphique ci-dessous a été fait par quelqu'un qui veut faire croire que la chaîne « explose ». Trouve le `piege`
(`"axe tronqué"`, `"causalité"` ou `"biais"`), calcule la vraie `hausse_pct` entre janvier et juin (arrondie à 1 décimale)
et **corrige le graphique** pour qu'il soit honnête (l'axe vertical doit partir de zéro, et le titre dire la vérité).
Enfin, réponds au quiz : pour chaque affirmation, `"causalité"`, `"graphique"` ou `"biais"` ?

- `pompiers` : « Les villes qui ont le plus de pompiers ont le plus d'incendies : réduisons les pompiers ! »
- `courbe_plafond` : « La température est passée de 14 à 16 °C : sur le graphique, la courbe grimpe jusqu'au plafond. »
- `sondage_club` : « Sondage sur le site d'un club de foot : 90 % des gens adorent le foot. »

<details><summary>Indice</summary>

`(abonnes[-1] / abonnes[0] - 1) * 100` ; `ax.set_ylim(0, ...)`. Pour le quiz : variable cachée (la taille de la ville) ? axe tronqué ? échantillon non représentatif ?

</details>

In [ ]:
mois = ["Jan", "Fév", "Mar", "Avr", "Mai", "Juin"]
abonnes = [1020, 1035, 1028, 1050, 1062, 1071]

# À toi
piege = "..."
hausse_pct = None
quiz = {"pompiers": "?", "courbe_plafond": "?", "sondage_club": "?"}

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(mois, abonnes, color="tab:red")
ax.set_ylim(1000, 1080)               # ← c'est ici que ça ment
ax.set_title("La chaîne explose !")
ax.set_ylabel("Abonnés")
plt.show()

In [ ]:
verifier("Exercice 10 · le piège", piege == "axe tronqué")
verifier("Exercice 10 · la vraie hausse", hausse_pct == 5.0)
verifier("Exercice 10 · graphique honnête", ax.get_ylim()[0] == 0)
verifier("Exercice 10 · quiz", quiz == {"pompiers": "causalité", "courbe_plafond": "graphique", "sondage_club": "biais"})

<details><summary>Solution</summary>

```python
mois = ["Jan", "Fév", "Mar", "Avr", "Mai", "Juin"]
abonnes = [1020, 1035, 1028, 1050, 1062, 1071]

piege = "axe tronqué"
hausse_pct = round((abonnes[-1] / abonnes[0] - 1) * 100, 1)
quiz = {"pompiers": "causalité",        # la variable cachée : la taille de la ville
        "courbe_plafond": "graphique",  # axe tronqué entre 14 et 16
        "sondage_club": "biais"}        # échantillon non représentatif

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(mois, abonnes, color="tab:blue")
ax.set_ylim(0, 1200)                   # l'axe part de zéro
ax.set_title(f"Les abonnés progressent doucement : +{hausse_pct} % en 6 mois")
ax.set_ylabel("Abonnés")
plt.show()
```

</details>

## Exercice 11 ⭐⭐⭐ · Plotly, en une ligne

Trois graphiques interactifs avec `plotly.express` :

1. `fig1` : nuage de points `total_bill` × `tip` de `tips`, une couleur par `day`, `hover_data=["time", "smoker"]`,
   avec un titre qui donne la réponse ;
2. `fig2` : boîtes à moustaches de `tip` par `day`, colorées par `smoker`, un sous-graphique par `time` (`facet_col`) ;
3. `fig3` : histogramme de `release_year` de `netflix`, coloré par `type`.

Résultat attendu : trois figures qui s'affichent et sur lesquelles tu peux survoler, zoomer, cliquer sur la légende.

<details><summary>Indice</summary>

`px.scatter(tips, x=..., y=..., color=..., hover_data=[...], title=...)`, `px.box(..., facet_col="time")`, `px.histogram(netflix, x="release_year", color="type")`.

</details>

In [ ]:
# À toi
fig1 = None   # px.scatter(...)
fig2 = None   # px.box(...)
fig3 = None   # px.histogram(...)
for fig in [fig1, fig2, fig3]:
    if fig is not None:
        fig.show()

In [ ]:
verifier("Exercice 11 · nuage de points (4 jours)", fig1 is not None and all(t.type == "scatter" for t in fig1.data) and len(fig1.data) == 4)
verifier("Exercice 11 · boîtes avec facettes", fig2 is not None and fig2.data[0].type == "box" and len(fig2.data) >= 2 and "xaxis2" in fig2.layout)
verifier("Exercice 11 · histogramme films / séries", fig3 is not None and fig3.data[0].type == "histogram" and len(fig3.data) == 2)

<details><summary>Solution</summary>

```python
fig1 = px.scatter(tips, x="total_bill", y="tip", color="day", hover_data=["time", "smoker"],
                  title="Le pourboire suit l'addition, quel que soit le jour", labels={"total_bill": "Addition ($)", "tip": "Pourboire ($)"})
fig2 = px.box(tips, x="day", y="tip", color="smoker", facet_col="time", title="Pourboires par jour, fumeurs ou non, midi et soir")
fig3 = px.histogram(netflix, x="release_year", color="type", title="Le catalogue Netflix est surtout fait de titres récents")
for fig in [fig1, fig2, fig3]:
    if fig is not None:
        fig.show()
```

</details>

## Exercice 12 ⭐⭐⭐ · Défi : ton pitch pour Netflix

Ton client est le responsable du catalogue Netflix. La colonne `listed_in` contient plusieurs genres séparés par
`", "`, et `country` plusieurs pays. Calcule `top_genres` (les 5 genres les plus fréquents, une Series), `genre_numero_1`,
et `top_pays` (les 3 pays les plus fréquents, une Series ; ignore les valeurs vides). Fais-en un graphique Plotly `fig_pitch`
(`px.bar`) avec un titre qui donne la réponse, puis remplis le `pitch` : constat, preuve, recommandation et piège vérifié.

Résultat attendu : un genre numéro 1 sans surprise pour un catalogue mondial, trois pays, un graphique et un pitch complet
(chaque case fait au moins 20 caractères).

<details><summary>Indice</summary>

`netflix["listed_in"].str.split(", ").explode().value_counts().head(5)` ; pareil pour `country` après `.dropna()` ; `px.bar(x=top_genres.index, y=top_genres.values, title=...)`.

</details>

In [ ]:
# À toi
top_genres = None
genre_numero_1 = None
top_pays = None
fig_pitch = None
pitch = {
    "client": "le responsable du catalogue Netflix",
    "constat": "...",
    "preuve": "... (quel graphique, quel chiffre)",
    "recommandation": "...",
    "piege_verifie": "... (axe à zéro ? causalité ? biais ?)",
}
print(top_genres)
print(top_pays)
if fig_pitch is not None:
    fig_pitch.show()
for etape, texte in pitch.items():
    print(f"{etape.upper():16s} {texte}")

In [ ]:
verifier("Défi · top genres", top_genres is not None and len(top_genres) == 5 and genre_numero_1 == "International Movies" and int(top_genres.iloc[0]) == 2437)
verifier("Défi · top pays", top_pays is not None and list(top_pays.index[:3]) == ["United States", "India", "United Kingdom"])
verifier("Défi · graphique Plotly", fig_pitch is not None and fig_pitch.data[0].type == "bar")
verifier("Défi · pitch complet", all(isinstance(v, str) and len(v) >= 20 and not v.startswith("...") for v in pitch.values()))

<details><summary>Solution</summary>

```python
top_genres = netflix["listed_in"].str.split(", ").explode().value_counts().head(5)
genre_numero_1 = top_genres.index[0]
top_pays = netflix["country"].dropna().str.split(", ").explode().value_counts().head(3)
fig_pitch = px.bar(x=top_genres.index, y=top_genres.values, labels={"x": "Genre", "y": "Nombre de titres"},
                   title="Les films internationaux et les drames dominent le catalogue")
pitch = {
    "client": "le responsable du catalogue Netflix",
    "constat": "Le catalogue est dominé par les films internationaux (2 437 titres) et les drames (2 106) ; les documentaires sont 3 fois moins nombreux.",
    "preuve": "Le graphique en barres des 5 genres les plus fréquents, axe à zéro : International Movies 2 437, Dramas 2 106, Comedies 1 471.",
    "recommandation": "Renforcer les documentaires et les contenus hors États-Unis / Inde / Royaume-Uni, sous-représentés par rapport à l'audience mondiale.",
    "piege_verifie": "Axe vertical à zéro ; un titre peut avoir plusieurs genres (les comptes se recoupent) ; 'nombreux' ne veut pas dire 'regardés' : pas de causalité.",
}
print(top_genres)
print(top_pays)
fig_pitch.show()
for etape, texte in pitch.items():
    print(f"{etape.upper():16s} {texte}")
```

</details>

## Bravo !

Tu sais poser une question précise, repérer une tendance, une anomalie et une corrélation, éviter les trois pièges
(causalité, axe tronqué, biais) et faire parler un graphique interactif. Pour aller plus loin : refais le défi avec le
dataset de ton choix de la séance 2, et présente ton pitch en 2 minutes chrono à quelqu'un qui joue le client.
